# ESMBiologist Demo

Zero-shot peptide scoring with ESM-2 embeddings.

In [1]:
import sys, os

project_root = os.path.abspath(os.path.join(os.path.dirname("__file__"), "..", ".."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from peptide_pipeline.biologist.esm_biologist import ESMBiologist
import transformers

/home/arthur/Documents/M2/pfe/multi-expert-peptide-pipeline/venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Define peptides

In [5]:
REFERENCE_PEPTIDE = "IKQLLHFFQRF"

candidate_peptides = [
    "RLLRRLLRRLLRRLLRRLLR",
    "LKRFLKWFKRF",
    "RLLKRFKHLFK",
    "TILQSLKNIFK",
    "FGTGVKRKAQCDLKSK",
    "AVVKVPLKKFKSIRETMKEKGLLEDF",
    "FFGHLFKLATKIIPSLFQ",
]

print(f"Reference : {REFERENCE_PEPTIDE}")
print(f"Candidates: {len(candidate_peptides)} peptides")
for i in range(min(len(candidate_peptides)-1, 10)):
    print(f"  {candidate_peptides[i]}")

Reference : IKQLLHFFQRF
Candidates: 7 peptides
  RLLRRLLRRLLRRLLRRLLR
  LKRFLKWFKRF
  RLLKRFKHLFK
  TILQSLKNIFK
  FGTGVKRKAQCDLKSK
  AVVKVPLKKFKSIRETMKEKGLLEDF


## Instantiate ESMBiologist

In [6]:
biologist = ESMBiologist(
    reference_peptide=REFERENCE_PEPTIDE,
    model_name="facebook/esm2_t33_650M_UR50D",
)
print(biologist)

Loading weights: 100%|██████████| 566/566 [00:00<00:00, 1077.68it/s, Materializing param=encoder.layer.32.output.dense.weight]                      
EsmModel LOAD REPORT from: facebook/esm2_t33_650M_UR50D
Key                         | Status     | 
----------------------------+------------+-
lm_head.bias                | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
esm.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
pooler.dense.weight         | MISSING    | 
pooler.dense.bias           | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


ESMBiologist(model='facebook/esm2_t33_650M_UR50D', device='cuda', reference='IKQLLHFFQRF...')


## score_peptides — cosine similarity to the reference

Scores are in **[0, 1]**: 1.0 = identical embedding, 0.5 = orthogonal, 0.0 = antipodal.

In [10]:
# Score all candidates
scores = biologist.score_peptides(candidate_peptides)

print(f"\n{'Peptide':<30} {'Score':>7}")
print("-" * 40)
for pep, sc in sorted(zip(candidate_peptides, scores), key=lambda x: -x[1]):
    print(f"{pep:<30} {sc:.4f}")


Peptide                          Score
----------------------------------------
RLLKRFKHLFK                    0.9919
TILQSLKNIFK                    0.9899
LKRFLKWFKRF                    0.9892
FGTGVKRKAQCDLKSK               0.9839
AVVKVPLKKFKSIRETMKEKGLLEDF     0.9767
FFGHLFKLATKIIPSLFQ             0.9505
RLLRRLLRRLLRRLLRRLLR           0.8870


## predict_activity with an alternative reference

In [13]:
TEMP_REFERENCE = "RLLRRLLRRLLRRLLRRLLR"

activity_scores = biologist.predict_activity(candidate_peptides, context=TEMP_REFERENCE)

print(f"{'Peptide':<30} {'Activity':>7}")
print("-" * 40)
for pep, sc in sorted(zip(candidate_peptides, activity_scores), key=lambda x: -x[1]):
    print(f"{pep:<30} {sc:.4f}")

Peptide                        Activity
----------------------------------------
RLLRRLLRRLLRRLLRRLLR           1.0000
LKRFLKWFKRF                    0.8997
RLLKRFKHLFK                    0.8958
TILQSLKNIFK                    0.8843
FGTGVKRKAQCDLKSK               0.8797
AVVKVPLKKFKSIRETMKEKGLLEDF     0.8712
FFGHLFKLATKIIPSLFQ             0.8606
